In [52]:
%%file ./src/av.cpp

#include <iostream>
#include <fstream>
#include <string>
#include <sstream>

#include <ftxui/dom/elements.hpp>
#include <ftxui/screen/screen.hpp>

// ----------------------------------------------------------------------------
// CONSTANTS

constexpr std::string_view HELP_MESSAGE = "\
Usage: ./av <PHYLIP_FILE>\
";

// ----------------------------------------------------------------------------
// FUNCTIONS

/*
 * Return padded string
 */
std::string left_pad(const std::string& s, const int size)
{
	std::string padded;
	padded.append(size - s.size(), ' ');
	padded.append(s);
	return padded;
}


// ----------------------------------------------------------------------------
// MAIN

int main(int argc, const char *argv[])
{
	if (argc != 2) {
		std::cerr << HELP_MESSAGE << "\n";
		return 1;
	}

	const char* file = argv[1];

	// get number of entries, and sequence length, from 1st line
	int entries, seq_len;
	std::ifstream f(file);
	if (f.good()) {
		std::string first;
		getline(f, first);
		std::stringstream ss(first);
		ss >> entries >> seq_len;
	} else {
		std::cerr << "File (" << file << ") does not exist or can't be opened.\n";
		return 1;
	}

	// collect tuples of (name, sequence)
	std::vector< std::array<std::string, 2> > records(entries);
	std::string line;
	int max_name_len = 10;
	if (f.is_open()) {
		for (int i = 0; i < entries; ++i) {
			std::getline(f, line);
			std::stringstream ss(line);
			ss >> records[i][0] >> records[i][1];  // assign values
			if (records[i][0].size() > max_name_len) {
				max_name_len = records[i][0].size();  // for name padding
			}
		}
		f.close();
	} else {
		std::cerr << "File (" << file << ") appears to contain zero entries.\n";
		return 1;
	}

	// TUI display
	using namespace ftxui;
	std::vector<Element>  hboxes;
	hboxes.reserve(entries);
	for (int i = 0; i < entries; ++i) {
		hboxes.push_back(
				hbox({
					text(left_pad(records[i][0], max_name_len)) | border,
					text(records[i][1])                         | border | flex,
				})
		);
	}

	Element document = vbox(hboxes);
	
	// create full-width/-height screen, limited to 80 wide
	document = document | size(WIDTH, LESS_THAN, 80);
	auto screen = Screen::Create(
		Dimension::Full(), Dimension::Fit(document)  // W, H
	);
	Render(screen, document);
	screen.Print();

	return 0;
}

Overwriting ./src/av.cpp


In [53]:
%%bash

./run_cmake.sh

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.




-- ┌─ FTXUI options ─────────────────────
-- │ FTXUI_BUILD_DOCS         : OFF
-- │ FTXUI_BUILD_EXAMPLES     : OFF
-- │ FTXUI_BUILD_MODULES      : OFF
-- │ FTXUI_BUILD_TESTS        : OFF
-- │ FTXUI_BUILD_TESTS_FUZZER : OFF
-- │ FTXUI_CLANG_TIDY         : OFF
-- │ FTXUI_DEV_WARNINGS       : OFF
-- │ FTXUI_ENABLE_COVERAGE    : OFF
-- │ FTXUI_ENABLE_INSTALL     : ON
-- │ FTXUI_QUIET              : OFF
-- └─────────────────────────────────────
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/colin/self_learning/programming/cplusplus/alignment_viewer/build
[  9%] Built target screen
[ 62%] Built target dom
[ 97%] Built target component
[ 98%] Building CXX object CMakeFiles/av.dir/src/av.cpp.o
[100%] Linking CXX executable av
[100%] Built target av
[  9%] Built target screen
[ 62%] Built target dom
[ 97%] Built target component
[100%] Built target av


In [54]:
%%bash

./build/av ./sample/in.phylip

╭──────────╮╭──────────────────────────────────────────────────────────────────╮
│     Alpha││AACGTGG___CAT                                                     │
╰──────────╯╰──────────────────────────────────────────────────────────────────╯
╭──────────╮╭──────────────────────────────────────────────────────────────────╮
│      Beta││AAGG_CGCCACAC                                                     │
╰──────────╯╰──────────────────────────────────────────────────────────────────╯
╭──────────╮╭──────────────────────────────────────────────────────────────────╮
│     Gamma││CAGTTCGCCACAA                                                     │
╰──────────╯╰──────────────────────────────────────────────────────────────────╯
╭──────────╮╭──────────────────────────────────────────────────────────────────╮
│     Delta││GAGATTTC__CCT                                                     │
╰──────────╯╰──────────────────────────────────────────────────────────────────╯
╭──────────╮╭───────────────